In [ ]:
# Set random seed for reproducibility
import random
import numpy as np
import torch
import os

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix

In [ ]:
df = pd.read_csv("data/survey.csv")

In [ ]:
df

In [ ]:
# Drop meaningless columns not needed
cols_to_drop = []
for col in df.columns:
  if col == 'Timestamp' or col == 'comments' or col == 'state':
    cols_to_drop.append(col)
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [ ]:
print("\nMissing values per column before cleaning:\n")
print(df.isnull().sum().sort_values(ascending=False))

In [ ]:
# Fill missing values in 'self_employed' with 'No'
df['self_employed'] = df['self_employed'].fillna('No')

# Fill missing work_interfere entries with 'Unknown'
df['work_interfere'] = df['work_interfere'].fillna('Unknown')

In [ ]:
print("\nMissing values per column after cleaning:\n")
print(df.isnull().sum().sort_values(ascending=False))

In [ ]:
# Standardize Gender column (Male, Female, Other)
if 'Gender' in df.columns:
    def clean_gender(g):
        if pd.isna(g):
            return 'Other'
        g = g.strip().lower()
        if g in ['male', 'm', 'man', 'male-ish', 'maile', 'cis male', 'cis-man']:
            return 'Male'
        elif g in ['female', 'f', 'woman', 'femake', 'female (cis)', 'cis female']:
            return 'Female'
        else:
            return 'Other'
    df['Gender'] = df['Gender'].apply(clean_gender)

In [ ]:
# Verify the cleaned gender distribution
if 'Gender' in df.columns:
    print("\nGender counts after cleaning:")
    print(df['Gender'].value_counts())

In [ ]:
before = df.shape[0]
df.drop_duplicates(inplace=True)
after = df.shape[0]
print(f"\nRemoved {before - after} duplicate rows.")

In [ ]:
# Map ordered text responses to numeric scales
ordinal_mappings = {
    'work_interfere': {
        'Never': 0,
        'Rarely': 1,
        'Sometimes': 2,
        'Often': 3,
        'Unknown': 4
    },
    'leave': {
        'Very easy': 0,
        'Somewhat easy': 1,
        "Don't know": 2,
        'Somewhat difficult': 3,
        'Very difficult': 4
    },
    'benefits': {
        'Yes': 1,
        'No': 0,
        "Don't know": 2
    },
    'care_options': {
        'Yes': 1,
        'No': 0,
        "Not sure": 2
    },
    'anonymity': {
        'Yes': 1,
        'No': 0,
        "Don't know": 2
    },
    'family_history': {
        'Yes': 1,
        'No': 0
    },
    'self_employed': {
        'Yes': 1,
        'No': 0
    },
    'treatment': {
        'Yes': 1,
        'No': 0
    }
}

# Apply mappings where applicable
for col, mapping in ordinal_mappings.items():
    if col in df.columns:
        df[col] = df[col].map(mapping).fillna(df[col])

categorical_cols = df.select_dtypes(include=['object']).columns

In [ ]:
# NORMALIZATION
numeric_cols = []
categorical_cols = []

for col in df.columns:
    if col == 'treatment':
        continue
    if df[col].dtype == 'object':
        categorical_cols.append(col)
    elif len(df[col].unique()) < 15 and col not in ordinal_mappings:
        categorical_cols.append(col)
    else:
        numeric_cols.append(col)

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# Separate features and target
X = df.drop(columns=['treatment'])
y = df['treatment'].astype(int)

scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])


X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_val.shape}")
print(f"Testing set shape: {X_test.shape}")

In [ ]:
# Convert to tensors - Training set
X_train_cat = torch.tensor(X_train[categorical_cols].values, dtype=torch.long)
X_train_num = torch.tensor(X_train[numeric_cols].values, dtype=torch.float)
y_train_t = torch.tensor(y_train.values, dtype=torch.float).unsqueeze(1)

# Convert to tensors - Validation set
X_val_cat = torch.tensor(X_val[categorical_cols].values, dtype=torch.long)
X_val_num = torch.tensor(X_val[numeric_cols].values, dtype=torch.float)
y_val_t = torch.tensor(y_val.values, dtype=torch.float).unsqueeze(1)

# Convert to tensors - Test set
X_test_cat = torch.tensor(X_test[categorical_cols].values, dtype=torch.long)
X_test_num = torch.tensor(X_test[numeric_cols].values, dtype=torch.float)
y_test_t = torch.tensor(y_test.values, dtype=torch.float).unsqueeze(1)

train_ds = TensorDataset(X_train_cat, X_train_num, y_train_t)
val_ds = TensorDataset(X_val_cat, X_val_num, y_val_t)
test_ds = TensorDataset(X_test_cat, X_test_num, y_test_t)

# Create dataloaders
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=32)
test_dl = DataLoader(test_ds, batch_size=32)


In [ ]:
class NNWithEmbeddings(nn.Module):
  def __init__(self, embedding_sizes, n_numeric, hidden_dims=[16, 8], dropout_rate=0.45):
    super().__init__()

    self.embeddings = nn.ModuleList([nn.Embedding(categories, size)
          for categories, size in embedding_sizes
    ])
    emb_dim = sum([emb.embedding_dim for emb in self.embeddings])

    self.bn_num = nn.BatchNorm1d(n_numeric)

    all_input_dim = emb_dim + n_numeric
    layers = []
    for h in hidden_dims:
        layers.append(nn.Linear(all_input_dim, h))
        layers.append(nn.Dropout(dropout_rate))
        layers.append(nn.ReLU())

        all_input_dim = h
    layers.append(nn.Linear(hidden_dims[-1], 1))
    layers.append(nn.Sigmoid())

    self.model = nn.Sequential(*layers)


  def forward(self, x_cat, x_num):
    embeddings = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
    x = torch.cat(embeddings, dim=1)
    x_num = self.bn_num(x_num)
    x = torch.cat([x, x_num], dim=1)
    return self.model(x)

In [ ]:
embedding_sizes = []
for col in categorical_cols:
    n_cat = int(df[col].max()) + 1
    emb_dim = max(2, min(50, (n_cat + 1) // 2))
    embedding_sizes.append((n_cat, emb_dim))

In [ ]:
from itertools import product

param_grid_regularized = {
    'lr': [0.0001, 0.001, 0.01],
    'hidden_dims': [[16, 8], [32, 16]],  
    'dropout_rate': [0.3, 0.4, 0.5],  
    'weight_decay': [1e-4, 1e-3, 1e-2]  
}

grid_search_results_reg = []
best_val_acc_reg = 0.0
best_params_reg = None
best_model_state_reg = None

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 125
patience = 15

print(f"Starting Regularized Grid Search with {len(param_grid_regularized['lr']) * len(param_grid_regularized['hidden_dims']) * len(param_grid_regularized['dropout_rate']) * len(param_grid_regularized['weight_decay'])} combinations")
print("=" * 80)

for lr, hidden_dims, dropout_rate, weight_decay in product(
    param_grid_regularized['lr'], 
    param_grid_regularized['hidden_dims'], 
    param_grid_regularized['dropout_rate'],
    param_grid_regularized['weight_decay']
):
    print(f"\nTesting: lr={lr}, hidden_dims={hidden_dims}, dropout_rate={dropout_rate}, weight_decay={weight_decay}")
    
    model = NNWithEmbeddings(embedding_sizes, len(numeric_cols), 
                            hidden_dims=hidden_dims, dropout_rate=dropout_rate)
    model.to(device)
    
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCELoss()
    
    train_losses = []
    val_losses = []
    val_accuracies = []
    best_val_loss = float('inf')
    patience_counter = 0
    best_epoch = 0
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for x_cat, x_num, y in train_dl:
            x_cat, x_num, y = x_cat.to(device, non_blocking=True), x_num.to(device, non_blocking=True), y.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            y_pred = model(x_cat, x_num)
            loss = criterion(y_pred, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        avg_train_loss = running_loss / len(train_dl)
        train_losses.append(avg_train_loss)
        
        model.eval()
        with torch.no_grad():
            val_loss, correct, total = 0, 0, 0
            for x_cat, x_num, y in val_dl:
                x_cat, x_num, y = x_cat.to(device), x_num.to(device), y.to(device)
                y_pred = model(x_cat, x_num)
                loss = criterion(y_pred, y)
                val_loss += loss.item()
                
                preds = (y_pred > 0.5).float()
                correct += (preds == y).sum().item()
                total += y.size(0)
        
        avg_val_loss = val_loss / len(val_dl)
        val_acc = correct / total
        
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_acc)
        
        # Early stopping
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_epoch = epoch + 1
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            break
    
    final_val_acc = val_accuracies[-1]
    final_val_loss = val_losses[-1]
    
    grid_search_results_reg.append({
        'lr': lr,
        'hidden_dims': hidden_dims,
        'dropout_rate': dropout_rate,
        'weight_decay': weight_decay,
        'val_accuracy': final_val_acc,
        'val_loss': final_val_loss,
        'train_loss': train_losses[-1],
        'best_epoch': best_epoch,
        'epochs_trained': len(train_losses)
    })
    
    print(f"  Final Val Acc: {final_val_acc:.4f}, Final Val Loss: {final_val_loss:.4f}, Best Epoch: {best_epoch}")
    
    # Track best model
    if final_val_acc > best_val_acc_reg:
        best_val_acc_reg = final_val_acc
        best_params_reg = {'lr': lr, 'hidden_dims': hidden_dims, 'dropout_rate': dropout_rate, 'weight_decay': weight_decay}
        best_model_state_reg = model.state_dict().copy()

print("\n" + "=" * 80)
print("Regularized Grid Search Complete!")
print(f"\nBest Parameters:")
print(f"  Learning Rate: {best_params_reg['lr']}")
print(f"  Hidden Dimensions: {best_params_reg['hidden_dims']}")
print(f"  Dropout Rate: {best_params_reg['dropout_rate']}")
print(f"  Weight Decay: {best_params_reg['weight_decay']}")
print(f"  Best Validation Accuracy: {best_val_acc_reg:.4f}")


In [ ]:
# Load the best model from grid search
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if best_params_reg is not None and best_model_state_reg is not None:
    model = NNWithEmbeddings(embedding_sizes, len(numeric_cols), 
                            hidden_dims=best_params_reg['hidden_dims'], 
                            dropout_rate=best_params_reg['dropout_rate'])
    model.load_state_dict(best_model_state_reg)
    model.to(device)
    
    print("Best model loaded successfully!")
    print(f"Parameters: {best_params_reg}")
    print(f"Validation Accuracy: {best_val_acc_reg:.4f}")

In [ ]:
# Re-train the best model to get training history for plotting

print("Re-training best model to capture training history...")
print(f"Using parameters: {best_params_reg}")

# Create fresh model with best hyperparameters
model_best = NNWithEmbeddings(embedding_sizes, len(numeric_cols), 
                                hidden_dims=best_params_reg['hidden_dims'], 
                                dropout_rate=best_params_reg['dropout_rate'])
model_best.to(device)

# Create optimizer with best hyperparameters
optimizer_best = optim.Adam(model_best.parameters(), 
                            lr=best_params_reg['lr'], 
                            weight_decay=best_params_reg['weight_decay'])
criterion = nn.BCELoss()

train_losses = []
val_losses = []
val_accuracies = []

epochs = 125
patience = 15
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(epochs):
    model_best.train()
    running_loss = 0.0
    
    for x_cat, x_num, y in train_dl:
        x_cat, x_num, y = x_cat.to(device, non_blocking=True), x_num.to(device, non_blocking=True), y.to(device, non_blocking=True)
        
        optimizer_best.zero_grad()
        y_pred = model_best(x_cat, x_num)
        loss = criterion(y_pred, y)
        loss.backward()
        optimizer_best.step()
        running_loss += loss.item()
    
    avg_train_loss = running_loss / len(train_dl)
    train_losses.append(avg_train_loss)
    
    model_best.eval()
    with torch.no_grad():
        val_loss, correct, total = 0, 0, 0
        for x_cat, x_num, y in val_dl:
            x_cat, x_num, y = x_cat.to(device), x_num.to(device), y.to(device)
            y_pred = model_best(x_cat, x_num)
            loss = criterion(y_pred, y)
            val_loss += loss.item()
            
            preds = (y_pred > 0.5).float()
            correct += (preds == y).sum().item()
            total += y.size(0)
    
    avg_val_loss = val_loss / len(val_dl)
    val_acc = correct / total
    
    val_losses.append(avg_val_loss)
    val_accuracies.append(val_acc)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
    else:
        patience_counter += 1
    
    if patience_counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

model = model_best
print(f"Final validation accuracy: {val_accuracies[-1]:.4f}")


In [ ]:
# Plot training and validation loss for best model
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(train_losses, label='Training Loss', color='blue')
ax.plot(val_losses, label='Validation Loss', color='orange')
ax.set_title('Training and Validation Loss Over Epochs (Best Model)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(val_accuracies, label='Validation Accuracy', color='green')
ax.set_title('Validation Accuracy Over Epochs (Best Model)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# EMBEDDINGS ABLATION STUDY

# Modified model class that can exclude specific embeddings
class NNWithEmbeddingsAblation(nn.Module):
    def __init__(self, embedding_sizes, n_numeric, hidden_dims=[32, 16], dropout_rate=0.3, excluded_indices=[]):
        super().__init__()
        
        self.embeddings = nn.ModuleList()
        self.embedding_indices = []
        
        for idx, (categories, size) in enumerate(embedding_sizes):
            if idx not in excluded_indices:
                self.embeddings.append(nn.Embedding(categories, size))
                self.embedding_indices.append(idx)
        
        emb_dim = sum([emb.embedding_dim for emb in self.embeddings])
        
        self.bn_num = nn.BatchNorm1d(n_numeric)
        self.excluded_indices = set(excluded_indices)
        
        all_input_dim = emb_dim + n_numeric
        layers = []
        for h in hidden_dims:
            layers.append(nn.Linear(all_input_dim, h))
            layers.append(nn.Dropout(dropout_rate))
            layers.append(nn.ReLU())
            all_input_dim = h
        layers.append(nn.Linear(hidden_dims[-1], 1))
        layers.append(nn.Sigmoid())
        
        self.model = nn.Sequential(*layers)
    
    def forward(self, x_cat, x_num):
        embeddings = []
        for i, emb in enumerate(self.embeddings):
            orig_idx = self.embedding_indices[i]
            embeddings.append(emb(x_cat[:, orig_idx]))
        
        if embeddings:
            x = torch.cat(embeddings, dim=1)
        else:
            x = torch.empty(x_cat.size(0), 0, device=x_cat.device)
        
        x_num = self.bn_num(x_num)
        x = torch.cat([x, x_num], dim=1)
        return self.model(x)

ablation_results = []

# Use best hyperparameters from grid search
best_lr = best_params_reg['lr']
best_hidden_dims = best_params_reg['hidden_dims']
best_dropout_rate = best_params_reg['dropout_rate']
best_weight_decay = best_params_reg['weight_decay']

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 125
patience = 15

val_accuracies_baseline = val_accuracies
val_losses_baseline = val_losses
train_losses_baseline = train_losses

model.eval()
test_correct, test_total = 0, 0
test_loss_baseline = 0.0
criterion = nn.BCELoss()

with torch.no_grad():
    for x_cat, x_num, y in test_dl:
        x_cat, x_num, y = x_cat.to(device), x_num.to(device), y.to(device)
        y_pred = model(x_cat, x_num)
        loss = criterion(y_pred, y)
        test_loss_baseline += loss.item()
        
        preds = (y_pred > 0.5).float()
        test_correct += (preds == y).sum().item()
        test_total += y.size(0)

test_acc_baseline = test_correct / test_total
test_loss_baseline = test_loss_baseline / len(test_dl)

ablation_results.append({
    'excluded_feature': 'None (Baseline)',
    'excluded_index': None,
    'val_accuracy': val_accuracies_baseline[-1],
    'val_loss': val_losses_baseline[-1],
    'test_accuracy': test_acc_baseline,
    'test_loss': test_loss_baseline,
    'best_epoch': len(val_accuracies_baseline),
    'train_loss': train_losses_baseline[-1]
})


In [ ]:
# Test removing each embedding one at a time
for idx, feature_name in enumerate(categorical_cols):    
    # Create model without this embedding
    model_abl = NNWithEmbeddingsAblation(embedding_sizes, len(numeric_cols), 
                                         hidden_dims=best_hidden_dims, 
                                         dropout_rate=best_dropout_rate,
                                         excluded_indices=[idx])
    model_abl.to(device)
    
    optimizer_abl = optim.Adam(model_abl.parameters(), lr=best_lr, weight_decay=best_weight_decay)
    criterion = nn.BCELoss()
    
    train_losses_abl = []
    val_losses_abl = []
    val_accuracies_abl = []
    best_val_loss_abl = float('inf')
    patience_counter_abl = 0
    
    for epoch in range(epochs):
        model_abl.train()
        running_loss = 0.0
        
        for x_cat, x_num, y in train_dl:
            x_cat, x_num, y = x_cat.to(device, non_blocking=True), x_num.to(device, non_blocking=True), y.to(device, non_blocking=True)
            
            optimizer_abl.zero_grad()
            y_pred = model_abl(x_cat, x_num)
            loss = criterion(y_pred, y)
            loss.backward()
            optimizer_abl.step()
            running_loss += loss.item()
        
        avg_train_loss = running_loss / len(train_dl)
        train_losses_abl.append(avg_train_loss)
        
        model_abl.eval()
        with torch.no_grad():
            val_loss, correct, total = 0, 0, 0
            for x_cat, x_num, y in val_dl:
                x_cat, x_num, y = x_cat.to(device), x_num.to(device), y.to(device)
                y_pred = model_abl(x_cat, x_num)
                loss = criterion(y_pred, y)
                val_loss += loss.item()
                
                preds = (y_pred > 0.5).float()
                correct += (preds == y).sum().item()
                total += y.size(0)
        
        avg_val_loss = val_loss / len(val_dl)
        val_acc = correct / total
        
        val_losses_abl.append(avg_val_loss)
        val_accuracies_abl.append(val_acc)
        
        if avg_val_loss < best_val_loss_abl:
            best_val_loss_abl = avg_val_loss
            patience_counter_abl = 0
        else:
            patience_counter_abl += 1
        
        if patience_counter_abl >= patience:
            break
    
    # Test on test set
    model_abl.eval()
    test_correct, test_total = 0, 0
    test_loss_abl = 0.0
    with torch.no_grad():
        for x_cat, x_num, y in test_dl:
            x_cat, x_num, y = x_cat.to(device), x_num.to(device), y.to(device)
            y_pred = model_abl(x_cat, x_num)
            loss = criterion(y_pred, y)
            test_loss_abl += loss.item()
            
            preds = (y_pred > 0.5).float()
            test_correct += (preds == y).sum().item()
            test_total += y.size(0)
    
    test_acc_abl = test_correct / test_total
    test_loss_abl = test_loss_abl / len(test_dl)
    
    # Calculate difference from baseline
    val_acc_diff = val_accuracies_abl[-1] - val_accuracies_baseline[-1]
    test_acc_diff = test_acc_abl - test_acc_baseline
    
    ablation_results.append({
        'excluded_feature': feature_name,
        'excluded_index': idx,
        'val_accuracy': val_accuracies_abl[-1],
        'val_loss': val_losses_abl[-1],
        'test_accuracy': test_acc_abl,
        'test_loss': test_loss_abl,
        'best_epoch': len(val_accuracies_abl),
        'train_loss': train_losses_abl[-1],
        'val_acc_diff': val_acc_diff,
        'test_acc_diff': test_acc_diff
    })


In [ ]:
# Summarize and visualize ablation results
ablation_df = pd.DataFrame(ablation_results)

ablation_df_sorted = ablation_df.sort_values('val_accuracy', ascending=False)

print("\n" + "=" * 80)
print("ABLATION RESULTS SUMMARY")
print("=" * 80)
print("\nResults sorted by Validation Accuracy (best to worst):")
print(ablation_df_sorted[['excluded_feature', 'val_accuracy', 'test_accuracy', 'val_acc_diff', 'test_acc_diff']].to_string(index=False))

baseline_val_acc = ablation_df[ablation_df['excluded_feature'] == 'None (Baseline)']['val_accuracy'].values[0]
baseline_test_acc = ablation_df[ablation_df['excluded_feature'] == 'None (Baseline)']['test_accuracy'].values[0]

best_config = ablation_df.loc[ablation_df['val_accuracy'].idxmax()]
worst_config = ablation_df.loc[ablation_df['val_accuracy'].idxmin()]


In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for x_cat, x_num, y in test_dl:
        x_cat, x_num, y = x_cat.to(device), x_num.to(device), y.to(device)
        y_pred = model(x_cat, x_num)
        preds = (y_pred > 0.5).float()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['No Treatment', 'Treatment'], yticklabels=['No Treatment', 'Treatment'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Test Set')
plt.show()